# Step 1

## Парсятся страны с кодами и загружаются в БД

In [1]:
from os import getenv

import pandas as pd
from sqlalchemy import create_engine, text


In [2]:
engine = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL"))

# I get it from Wikipedia and save country codes and the U.S. state codes tables in the database.
And make those codes the primary keys.

### Table of all countries and Alpha 3 codes

In [24]:
url_countries = 'https://en.wikipedia.org/wiki/ISO_3166-1'

In [25]:
df_countries = pd.read_html(
    url_countries, match="Alpha-3 code", storage_options={"User-Agent": "Mozilla/5.0"}
)[0]

df_countries = df_countries.rename(
    columns={
        "English short name (using title case)": "country",
        "Alpha-2 code": "alpha2_code",
        "Alpha-3 code": "country_code",
        "Numeric code": "numeric_code",
        "Independent[b]": "independent",
    }
)

df_countries = df_countries.drop(columns=["Link to ISO 3166-2"])

In [26]:
df_countries

,country,alpha2_code,country_code,numeric_code,independent
0,Afghanistan[c],AF,AFG,4,Yes
1,Åland Islands,AX,ALA,248,No
2,Albania,AL,ALB,8,Yes
3,Algeria,DZ,DZA,12,Yes
4,American Samoa,AS,ASM,16,No
...,...,...,...,...,...
244,Wallis and Futuna,WF,WLF,876,No
245,Western Sahara[c],EH,ESH,732,No
246,Yemen,YE,YEM,887,Yes
247,Zambia,ZM,ZMB,894,Yes


Делаем очистку названия страны:
- `\[.*?\]` — находим любой текст в квадратных скобках ([c], [note 1], и т.д.)
- удаляем его
- `strip()` убираем лишние пробелы

In [ ]:
for country_code, row in df_countries.iterrows():
    if country_code != 'VGB' and country_code != 'VIR':
        df_countries["country"] = (
            df_countries["country"].str.replace(r"\[.*?\]", "", regex=True).str.strip()
        )

In [31]:
df_countries.head(2)

,country,alpha2_code,country_code,numeric_code,independent
0,Afghanistan,AF,AFG,4,Yes
1,Åland Islands,AX,ALA,248,No


## Загрузка df в БД

In [ ]:
df_countries.to_sql('country', engine, index=False)
with engine.connect() as connection:
    connection.execute(text("ALTER TABLE country ADD CONSTRAINT PK_country_code PRIMARY KEY (country_code)"))
    connection.commit()

In [37]:
assert df_countries["country_code"].notna().all()
assert df_countries["country_code"].is_unique

df_countries.to_sql(
    "countries",
    engine,
    index=False,
    if_exists="fail",
)

with engine.connect() as connection:
    connection.execute(
        text(
            "ALTER TABLE countries "
            "ADD CONSTRAINT pk_country_code PRIMARY KEY (country_code)"
        )
    )
    connection.commit()

# Table of all U.S. states in ISO format

In [ ]:
url_states = 'https://en.wikipedia.org/wiki/ISO_3166-2:US'
df_states = pd.read_html(url_states, storage_options={"User-Agent": "Mozilla/5.0"})[0]
df_states.columns = ['state_code', 'state_name', 'category']
df_states.head(2)

In [ ]:
df_states.to_sql('states', engine, index=False)
with engine.connect() as connection:
    connection.execute(text("ALTER TABLE states ADD CONSTRAINT PK_state_code PRIMARY KEY (state_code)"))
    connection.commit()